# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My label (`is_declining`, from w04) is binary and observed — a page either lost impressions in the second half of March or it didn't. Per the toolkit, that's "yes/no with an observed label," so I start with **Logistic Regression** (readable, gives a coefficient per signal) then **Random Forest** (can pick up interactions a linear model can't — e.g. "low CTR AND poor position" mattering more together than either alone, which is exactly the kind of interaction w02 argued a fixed rule can't capture).

Because the real use case is "which pages first?" with limited review capacity, I evaluate with **precision@20** (my chosen success metric from w02) and precision@50 for a second look, not accuracy.

I deliberately exclude `baseline_score` itself from the model's features — feeding the model the baseline's own output would let it just imitate the rule rather than discover anything. I also exclude staleness/`days_since_update`, since w04 already found it FALSE on this data.

In [1]:
%pip -q install duckdb scikit-learn
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(gsc_avg_position) AS avg_position_h1,
        COUNT(DISTINCT report_date) AS active_days_h1
    FROM {MAR}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
    HAVING impressions_h1 >= 20
""").df()
features["ctr_h1"] = features["clicks_h1"] / features["impressions_h1"] * 100

labels = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {MAR}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id
""").df()

data = features.merge(labels, on="content_hash_id", how="left")
data["impressions_h2"] = data["impressions_h2"].fillna(0)
data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)

dim_content = con.sql(f"""
    SELECT content_hash_id, content_type, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()
data = data.merge(dim_content, on="content_hash_id", how="left")

client_lookup = con.sql(f"SELECT DISTINCT content_hash_id, client_hash_id FROM {MAR}").df()
data = data.merge(client_lookup, on="content_hash_id", how="left")

data["position_tier"] = pd.cut(data["avg_position_h1"], bins=[0,3,10,20,9999],
                                labels=["1-3","4-10","11-20","21+"])
bucket2 = data.groupby("position_tier", observed=True).apply(
    lambda g: pd.Series({"weighted_ctr": g["clicks_h1"].sum() / g["impressions_h1"].sum() * 100})
)
data["expected_ctr"] = data["position_tier"].map(bucket2["weighted_ctr"].to_dict()).astype(float)
data["ctr_gap"] = data["expected_ctr"] - data["ctr_h1"]

VISIBLE_MIN_IMPRESSIONS = 100
is_underperform = (data["ctr_gap"] > 0) & (data["impressions_h1"] >= VISIBLE_MIN_IMPRESSIONS)
data["baseline_score"] = np.where(is_underperform, data["impressions_h1"] * data["ctr_gap"], 0)

print(data.shape)
print("declining rate:", data["is_declining"].mean().round(3))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 15)
declining rate: 0.291


/tmp/ipykernel_7427/883903326.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bucket2 = data.groupby("position_tier", observed=True).apply(


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,ctr_h1,impressions_h2,is_declining,content_type,content_updated_date,client_hash_id,position_tier,expected_ctr,ctr_gap,baseline_score
0,content_7a105f548d9c6916,4173.0,6.0,6.327311,15,0.143781,2350.0,1,keyword article,2026-07-06,client_73cda7b4e4f265ea,4-10,0.327585,0.183803,767.011097
1,content_a3ea9792f793ec72,245.0,0.0,3.906852,15,0.000000,208.0,0,keyword article,2026-05-18,client_73cda7b4e4f265ea,4-10,0.327585,0.327585,80.258260
2,content_36c36abc7650d7af,3705.0,3.0,6.473735,15,0.080972,1925.0,1,keyword article,2026-05-20,client_73cda7b4e4f265ea,4-10,0.327585,0.246613,913.701441
3,content_a7da352b73b02668,2440.0,8.0,7.259861,15,0.327869,2504.0,0,keyword article,2026-07-06,client_73cda7b4e4f265ea,4-10,0.327585,-0.000284,0.000000
4,content_1855a661b4d36130,240.0,1.0,3.860842,15,0.416667,189.0,1,keyword article,2026-05-18,client_73cda7b4e4f265ea,4-10,0.327585,-0.089082,0.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I split by `client_hash_id`, not by row, using a grouped split — content items from the same client share editorial style, template, and traffic patterns, so a row-level random split would leak a client's "personality" across train and test and inflate the score. This matches the grouped-split approach named in the data contract (w03). 75/25 split, random_state=42 fixed for reproducibility.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

model_df = data.dropna(subset=["avg_position_h1", "position_tier", "content_type"]).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Clients in both train and test (should be 0):", len(overlap))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train decline rate:", train_df["is_declining"].mean().round(3),
      "| Test decline rate:", test_df["is_declining"].mean().round(3))

Clients in both train and test (should be 0): 0
Train rows: 93389 | Test rows: 16193
Train decline rate: 0.294 | Test decline rate: 0.274


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same rows, three scorers compared: the w04 baseline rule, Logistic Regression, and Random Forest — evaluated with precision@20, precision@50, ROC-AUC, and the base rate.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score

numeric_features = ["impressions_h1", "clicks_h1", "avg_position_h1",
                     "active_days_h1", "ctr_h1", "ctr_gap", "expected_ctr"]
categorical_features = ["content_type"]

X_train = train_df[numeric_features + categorical_features]
X_test = test_df[numeric_features + categorical_features]
y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
]).fit(X_train, y_train)

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                    class_weight="balanced", random_state=42, n_jobs=-1)),
]).fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scores = {
    "baseline rule": test_df["baseline_score"].values,
    "logistic regression": logreg.predict_proba(X_test)[:, 1],
    "random forest": rf.predict_proba(X_test)[:, 1],
}

rows = []
for name, s in scores.items():
    rows.append({
        "method": name,
        "precision@20": round(precision_at_k(s, y_test.values, 20), 3),
        "precision@50": round(precision_at_k(s, y_test.values, 50), 3),
        "ROC-AUC": round(roc_auc_score(y_test, s), 3),
    })
rows.append({"method": "base rate (random)", "precision@20": round(y_test.mean(), 3),
             "precision@50": round(y_test.mean(), 3), "ROC-AUC": 0.5})

comparison = pd.DataFrame(rows)
comparison

,method,precision@20,precision@50,ROC-AUC
0,baseline rule,0.000,0.100,0.502
1,logistic regression,0.150,0.420,0.559
2,random forest,0.150,0.340,0.589
3,base rate (random),0.274,0.274,0.500


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Feature importance from permutation importance on the random forest (shuffling each feature and measuring the ROC-AUC drop), plus three concrete wrong cases to see what the model leans on and where it breaks.

In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42,
                               scoring="roc_auc", n_jobs=-1)

feature_names = numeric_features + categorical_features
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
}).sort_values("importance_mean", ascending=False)

print(importance_df.head(5))

test_df = test_df.copy()
test_df["rf_proba"] = rf.predict_proba(X_test)[:, 1]
test_df["error"] = np.abs(test_df["rf_proba"] - test_df["is_declining"])

wrong_cases = test_df.sort_values("error", ascending=False).head(3)[
    ["content_hash_id", "impressions_h1", "avg_position_h1", "ctr_h1",
     "rf_proba", "is_declining"]
]
wrong_cases

           feature  importance_mean
3   active_days_h1         0.021901
2  avg_position_h1         0.018344
4           ctr_h1         0.012253
0   impressions_h1         0.011652
7     content_type         0.006084


,content_hash_id,impressions_h1,avg_position_h1,ctr_h1,rf_proba,is_declining
41057,content_bf89a688c0ec7cd7,68.0,41.290635,0.0,0.033090,1
105972,content_c098a7a69022061d,46.0,28.314863,0.0,0.033465,1
41083,content_6011e836cf18643a,34.0,39.481818,0.0,0.034126,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.